In [1]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
import numpy as np
import open3d as o3d
import requests


Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Chargement de l'image (commune aux deux procédés)

In [2]:
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\test_plan_batiment.png"
#IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\test_original_RGB.png"
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\screenshot_plan_2.png"
image = Image.open(IMAGE_PATH).convert("RGB")
# Redimensionnement pour GLPN (multiples de 32) ; utilisé tel quel en procédé 1
new_height = 480 if image.height > 480 else image.height
new_height -= (new_height % 32)
new_width = int(new_height * image.width / image.height)
new_width = new_width + (32 - new_width % 32) if new_width % 32 else new_width
image = image.resize((new_width, new_height))

## Procédé 1 : Image telle quelle → nuage de points
Utilisation de l'image sans estimation de profondeur : couleur = image, profondeur = niveaux de gris (luminance).

In [3]:
def create_point_cloud_from_image(color_image, depth_image, intrinsic, depth_scale=1000.0, remove_white=True):
    """
    Nuage de points à partir d'une image couleur et d'une carte de profondeur (numpy).
    Utilise Open3D RGBDImage.create_from_color_and_depth + PointCloud.create_from_rgbd_image.
    Si remove_white=True, enlève les points complètement blancs (RGB ≈ 1,1,1).
    """
    h, w = depth_image.shape
    color_o3d = o3d.geometry.Image(np.asarray(color_image, dtype=np.uint8))
    depth_o3d = o3d.geometry.Image(depth_image.astype(np.float32))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=depth_scale, depth_trunc=1000.0, convert_rgb_to_intensity=False
    )
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    if remove_white and pcd.has_colors():
        colors = np.asarray(pcd.colors)
        not_white = np.any(colors < 0.65, axis=1)
        indices = np.where(not_white)[0]
        pcd = pcd.select_by_index(indices)
    return pcd


def create_flat_point_cloud_from_image(color_image, depth_scale=1000.0):
    """
    Nuage de points aplati (plan) à partir d'une image : profondeur constante + intrinsèques
    orthographiques. Utilise PointCloud.create_from_rgbd_image (doc Open3D geometry).
    """
    print(color_image.size)
    w, h = color_image.size
    depth_flat = np.full((h, w), depth_scale, dtype=np.float32)

    fx_ortho = fy_ortho = 1e6
    intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx_ortho, fy_ortho, w / 2, h / 2)
    return create_point_cloud_from_image(color_image, depth_flat, intrinsic, depth_scale=depth_scale)

f = create_flat_point_cloud_from_image(image)


(960, 480)


Reconstruction Poisson


In [4]:
import open3d as o3d


# Estimation des normales
points.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.05, max_nn=30
    )
)

#Reconstruction Poisson
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    points, depth=9
)

#Suppression des triangles peu fiables
import numpy as np
densities = np.asarray(densities)
vertices_to_keep = densities > np.quantile(densities, 0.05)
mesh = mesh.select_by_index(np.where(vertices_to_keep)[0])
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()

o3d.visualization.draw_geometries([mesh], window_name="Reconstruction Poisson")


NameError: name 'points' is not defined

Technique BIM

In [5]:
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import LinearRegression

def extract_lines(points, min_points=50):
    remaining = points.copy()
    lines = []

    while len(remaining) > min_points:
        X = remaining[:, 0].reshape(-1, 1)
        y = remaining[:, 1]

        ransac = RANSACRegressor(
            LinearRegression(),
            residual_threshold=0.02,
            min_samples=50
        )

        try:
            ransac.fit(X, y)
        except:
            break

        inlier_mask = ransac.inlier_mask_
        if inlier_mask.sum() < min_points:
            break

        line_points = remaining[inlier_mask]
        lines.append(line_points)

        remaining = remaining[~inlier_mask]

    return lines

np_points = np.asarray(points.points)
points_2d = np_points[:, :2]  # requis pour les cellules suivantes (DBSCAN, etc.)
lines = extract_lines(np_points)


NameError: name 'points' is not defined

In [6]:
from sklearn.cluster import DBSCAN

# vecteurs directionnels locaux (différences entre voisins)
vectors = points_2d[1:] - points_2d[:-1]
angles = np.arctan2(vectors[:, 1], vectors[:, 0])
angles = np.mod(angles, np.pi)  # 0..pi

angles = angles.reshape(-1, 1)

labels = DBSCAN(
    eps=np.deg2rad(5),
    min_samples=100
).fit_predict(angles)


NameError: name 'points_2d' is not defined

In [ ]:
lines = []

for label in set(labels):
    if label == -1:
        continue

    mask = labels == label
    group_points = points_2d[:-1][mask]

    if len(group_points) < 100:
        continue

    # PCA pour cette orientation
    mean = group_points.mean(axis=0)
    U, S, Vt = np.linalg.svd(group_points - mean)
    direction = Vt[0]

    t = (group_points - mean) @ direction
    ordered = group_points[np.argsort(t)]

    lines.append(ordered)


NameError: name 'labels' is not defined

In [ ]:
from shapely.geometry import LineString

wall_thickness = 1  # mètres
wall_polygons = []

for line_pts in lines:
    line = LineString(line_pts)
    wall = line.buffer(
        wall_thickness / 2,
        cap_style=2,  # murs plats
        join_style=2
    )
    wall_polygons.append(wall)


In [ ]:
from shapely.ops import unary_union

walls_2d = unary_union(wall_polygons)



In [ ]:
from shapely.geometry import LineString

test_line = LineString(points_2d[:50])
test_wall = test_line.buffer(1.0)

print(test_wall.area)


NameError: name 'points_2d' is not defined

In [ ]:
import geopandas as gpd
gdf = gpd.GeoDataFrame(geometry=[walls_2d])
ax = gdf.plot(figsize=(6, 6))
ax.set_aspect("equal")
gpd.GeoSeries([test_wall]).plot(ax = ax,cmap="Set1")
gpd.show_versions()

#gdf = gpd.GeoDataFrame(geometry=[walls_2d])




c:\Users\mvm\open3d_vision\.venv\Lib\site-packages\geopandas\plotting.py:962: UserWarning: The GeoSeries you are attempting to plot is composed of empty geometries. Nothing has been displayed.
  return plot_dataframe(data, *args, **kwargs)


NameError: name 'test_wall' is not defined

In [ ]:
def create_plane_mesh(plane_model, bbox, size_factor=1.2):
    """
    Crée un TriangleMesh représentant le plan détecté.
    
    Parameters:
    - plane_model: [a, b, c, d] où ax + by + cz + d = 0
    - bbox: Bounding box du nuage de points pour déterminer la taille du plan
    - size_factor: Facteur pour agrandir le plan par rapport à la bounding box
    
    Returns:
    - TriangleMesh représentant le plan
    """
    [a, b, c, d] = plane_model
    
    # Obtenir les dimensions de la bounding box
    extent = bbox.get_extent()
    center = bbox.get_center()
    
    # Créer un plan assez grand pour couvrir le nuage
    # Utiliser les deux plus grandes dimensions
    dims = sorted(extent, reverse=True)
    plane_size_x = dims[0] * size_factor
    plane_size_y = dims[1] * size_factor if len(dims) > 1 else dims[0] * size_factor
    
    # Créer un plan dans le plan XY puis le transformer
    # Points du plan dans le repère local
    half_x = plane_size_x / 2
    half_y = plane_size_y / 2
    
    # Créer 4 points formant un rectangle
    if abs(c) > 1e-6:  # Plan non horizontal
        # Calculer un point sur le plan (solution de ax + by + cz + d = 0)
        # Prendre x=0, y=0 => z = -d/c
        p0 = np.array([0, 0, -d/c])
        
        # Vecteurs dans le plan
        # Vecteur perpendiculaire à la normale dans le plan XY
        if abs(a) > 1e-6 or abs(b) > 1e-6:
            v1 = np.array([-b, a, 0])
            v1 = v1 / np.linalg.norm(v1) * half_x
            v2 = np.cross([a, b, c], v1)
            v2 = v2 / np.linalg.norm(v2) * half_y
        else:
            # Plan horizontal
            v1 = np.array([half_x, 0, 0])
            v2 = np.array([0, half_y, 0])
    else:
        # Plan horizontal (c ≈ 0)
        p0 = np.array([0, 0, center[2]])
        v1 = np.array([half_x, 0, 0])
        v2 = np.array([0, half_y, 0])
    
    # Déplacer le centre du plan au centre de la bounding box
    # Projeter le centre sur le plan
    if abs(c) > 1e-6:
        center_z = -(a * center[0] + b * center[1] + d) / c
        plane_center = np.array([center[0], center[1], center_z])
    else:
        plane_center = center
    
    # Créer les 4 coins du rectangle
    corners = np.array([
        plane_center - v1 - v2,
        plane_center + v1 - v2,
        plane_center + v1 + v2,
        plane_center - v1 + v2
    ])
    
    # Créer le mesh avec 2 triangles
    plane_mesh = o3d.geometry.TriangleMesh()
    plane_mesh.vertices = o3d.utility.Vector3dVector(corners)
    plane_mesh.triangles = o3d.utility.Vector3iVector([[0, 1, 2], [0, 2, 3]])
    plane_mesh.compute_vertex_normals()
    
    # Couleur semi-transparente verte pour le plan
    plane_mesh.paint_uniform_color([0, 1, 0])
    
    return plane_mesh

# Vérification que le nuage de points n'est pas vide
if len(points.points) == 0:
    print("Erreur : Le nuage de points est vide. Vérifiez le filtrage des points blancs.")
else:
    print(f"Nombre de points dans le nuage : {len(points.points)}")
    
    # Calcul d'un distance_threshold adapté à l'échelle du nuage
    # Utilisation d'un pourcentage de la taille de la bounding box
    bbox = points.get_axis_aligned_bounding_box()
    bbox_size = bbox.get_extent()
    # Utiliser 1% de la plus grande dimension comme seuil
    adaptive_threshold = max(bbox_size) * 0.01
    # Minimum de 0.001 pour éviter les valeurs trop petites
    distance_threshold = max(adaptive_threshold, 0.001)
    
    print(f"Distance threshold utilisé : {distance_threshold:.6f}")
    
    # Segmentation du plan
    plane_model, inliers = points.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=3,
        num_iterations=1000
    )
    
    [a, b, c, d] = plane_model
    print(f"Plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")
    print(f"Nombre de points inliers : {len(inliers)}")
    print(f"Nombre de points outliers : {len(points.points) - len(inliers)}")
    
    # Création des nuages inliers et outliers
    inlier_cloud = points.select_by_index(inliers)
    outlier_cloud = points.select_by_index(inliers, invert=True)
    
    # Création de la visualisation du plan
    plane_mesh = create_plane_mesh(plane_model, bbox)
    
    # Vérification que les nuages ne sont pas vides avant l'affichage
    geoms_to_display = []
    
    if len(inlier_cloud.points) > 0:
        inlier_cloud.paint_uniform_color([1.0, 0, 0])  # Rouge pour les inliers
        geoms_to_display.append(inlier_cloud)
        print("Nuage inliers créé avec succès")
    else:
        print("Attention : Le nuage inliers est vide")
    
    if len(outlier_cloud.points) > 0:
        outlier_cloud.paint_uniform_color([0, 0, 1.0])  # Bleu pour les outliers
        geoms_to_display.append(outlier_cloud)
        print("Nuage outliers créé avec succès")
    else:
        print("Info : Le nuage outliers est vide (normal pour un nuage de points plat)")
    
    # Ajouter le plan à la visualisation
    geoms_to_display.append(plane_mesh)
    print("Plan détecté ajouté à la visualisation (vert)")
    
    # Affichage seulement si au moins un nuage contient des points
    if len(geoms_to_display) > 0:
        # Calcul automatique des paramètres de visualisation basés sur le nuage
        center = bbox.get_center()
        extent = bbox.get_extent()
        max_extent = max(extent)
        
        # Paramètres de visualisation adaptés
        lookat = center
        front = [0, 0, -1]  # Vue de face
        up = [0, 1, 0]  # Orientation verticale
        zoom = 0.7
        
        print(f"\nAffichage de {len(geoms_to_display)} géométrie(s)...")
        o3d.visualization.draw_geometries(
            geoms_to_display,
            zoom=zoom,
            front=front,
            lookat=lookat,
            up=up,
            window_name="Segmentation du plan - Inliers (rouge), Outliers (bleu), Plan (vert)"
        )
    else:
        print("Erreur : Aucun nuage à afficher")

NameError: name 'points' is not defined

Technique KNN pour liaison des points sur un même plan

In [ ]:
# (old) def link_closest_points(pcd):
#     """
#     Crée un LineSet avec des arêtes reliant les points les plus proches.
#     """
#     pcd_tree = o3d.geometry.KDTreeFlann(pcd)
#     lines = []
#     for i in range(len(pcd.points)):
#         [k, idx, _] = pcd_tree.search_knn_vector_3d(pcd.points[i], 2)
#         if k != 0:
#             lines.append([i, idx[1]])
#     line_set = o3d.geometry.LineSet()
#     line_set.points = o3d.utility.Vector3dVector(pcd.points)
#     line_set.lines = o3d.utility.Vector2iVector(lines)
#     return line_set)
#new
def link_closest_points(pcd, debug=True):
    """
    Crée un LineSet avec des arêtes reliant chaque point à son plus proche voisin.
    Retourne (line_set, pcd_knn_red) : le LineSet et un nuage où les points concernés par le KNN sont en rouge.
    """
    pts = np.asarray(pcd.points)
    n = pts.shape[0]
    if n < 2:
        if debug:
            print("[link_closest_points] Trop peu de points:", n)
        return o3d.geometry.LineSet(), None

    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    lines = []
    for i in range(n):
        [k, idx, _] = pcd_tree.search_knn_vector_3d(pts[i], 3)
        if k >= 2 and idx[1] != i:
            lines.append([i, int(idx[1])])

    # Points concernés par le KNN (extrémités des arêtes) -> rouge
    concerned = set(i for line in lines for i in line)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else np.ones((n, 3)) * 0.7
    colors = np.copy(colors)
    colors[list(concerned)] = [1, 0, 0]
    pcd_knn_red = o3d.geometry.PointCloud()
    pcd_knn_red.points = o3d.utility.Vector3dVector(pts)
    pcd_knn_red.colors = o3d.utility.Vector3dVector(colors)

    if debug:
        print("[link_closest_points] Points:", n, "| Lignes:", len(lines), "| En rouge:", len(concerned))
        if len(lines) > 0:
            print("  Ex. lignes:", lines[:3])
            arr = np.array(lines)
            lens = np.linalg.norm(pts[arr[:, 1]] - pts[arr[:, 0]], axis=1)
            print("  Long. min/max:", lens.min(), "/", lens.max())

    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(pts)
    line_set.lines = o3d.utility.Vector2iVector(np.array(lines, dtype=np.int64))
    line_set.colors = o3d.utility.Vector3dVector(np.tile([[0.2, 0.2, 0.2]], (len(lines), 1)))
    return line_set, pcd_knn_red

#nuage aplati (plan) via Open3D create_from_rgbd_image + profondeur constante
img_np = np.array(image)
pcd_direct = create_flat_point_cloud_from_image(img_np, depth_scale=1)
ls_closest, pcd_knn_red = link_closest_points(pcd_direct,True)
# Nuage avec points concernés par le KNN en rouge ; lignes = plus proches voisins
geoms = [g for g in [pcd_knn_red, ls_closest] if g is not None] # un coalesce en fonctionnel
print(type(geoms[0]),type(geoms[1]))
o3d.visualization.draw_geometries([geoms[1]], window_name="KNN (points rouges + arêtes)")

1382400


TypeError: cannot unpack non-iterable int object

Duplication des deux plans de points et liaison avec des edges

In [ ]:
def extrude_point_cloud(pcd, wall_height=0.05):
    """
    Extrude en deux couches : plan inférieur (z=0) et plan supérieur (z=wall_height).
    Retourne un nuage avec 2*N points : [0..N-1] = base, [N..2N-1] = haut.
    """
    pts = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else np.ones((len(pts), 3)) * 0.7
    pts_top = pts + np.array([0, 0, wall_height])
    pts_all = np.vstack([pts, pts_top])
    col_all = np.vstack([colors, colors])
    pcd_ext = o3d.geometry.PointCloud()
    pcd_ext.points = o3d.utility.Vector3dVector(pts_all)
    pcd_ext.colors = o3d.utility.Vector3dVector(col_all)
 
    print(type(pts_top))
    return pcd_ext, pts_top,pts


def add_edges_between_layers(pcd_extruded):
    """
    À partir du nuage extrudé (2 couches : inférieur puis supérieur), crée un LineSet
    avec des arêtes reliant chaque point du plan inférieur au point correspondant du plan supérieur.
    """
    pts = np.asarray(pcd_extruded.points)
    #transforme pts en array numpy
    n = len(pts) // 2
    lines = np.array([[i, i + n] for i in range(n)], dtype=np.int32)
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(pts)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    if pcd_extruded.has_colors():
        line_set.colors = o3d.utility.Vector3dVector(np.tile([0.2, 0.2, 0.2], (len(lines), 1)))
    return line_set
extrude_point_cloud(pcd_direct, wall_height=0.00005)
# Extrusion : deux couches (base + haut)
pcd_all, pts_top,pts = extrude_point_cloud(pcd_direct, wall_height=0.00005)

def add_edges_between_layers(pts_top,pts):
    """
    À partir du nuage extrudé (2 couches : inférieur puis supérieur), crée un LineSet
    avec des arêtes reliant chaque point du plan inférieur au point correspondant du plan supérieur.
    """
    pcd_top = o3d.geometry.PointCloud()
    pcd_top.points = o3d.utility.Vector3dVector(pts_top)
    pcd_top.colors = o3d.utility.Vector3dVector(np.ones((len(pts_top), 3)) * 0.7)
    pcd_base = o3d.geometry.PointCloud()
    pcd_base.points = o3d.utility.Vector3dVector(pts)
    pcd_base.colors = o3d.utility.Vector3dVector(np.ones((len(pts), 3)) * 0.7)
    # Variable : nuage base+top avec une seule arête par paire (point inférieur i <-> point supérieur i)
    # Ordre des points : [0..n-1] = base, [n..2n-1] = top. Chaque ligne = (i, n+i) uniquement.
    pts_combined = np.vstack([pts, pts_top])
    n = len(pts)
    # Exactement n segments : segment k relie base[k] à top[k], pas d'autre liaison
    lines_correspond = np.zeros((n, 2), dtype=np.int64)
    lines_correspond[:, 0] = np.arange(n)       # indice plan inférieur
    lines_correspond[:, 1] = np.arange(n) + n    # indice plan supérieur correspondant
    pcd_base_top_with_edges = o3d.geometry.LineSet()
    pcd_base_top_with_edges.points = o3d.utility.Vector3dVector(pts_combined)
    pcd_base_top_with_edges.lines = o3d.utility.Vector2iVector(lines_correspond)
    # Dégradé : premières lignes sombres, dernières claires (une couleur par ligne)
    t = np.linspace(0, 1, n).reshape(-1, 1)
    gray = (0.15 + 0.75 * t).reshape(-1)
    line_colors = np.column_stack([gray, gray, gray])
    pcd_base_top_with_edges.colors = o3d.utility.Vector3dVector(line_colors)
    return pcd_base_top_with_edges


o3d.visualization.draw_geometries([add_edges_between_layers(pts_top,pts)], window_name="Procédé 1 - Extrusion (murs)")

# # Arêtes verticales (base -> haut)
# line_set_walls = add_edges_between_layers(pcd_walls)
# o3d.visualization.draw_geometries([line_set_walls], window_name="Murs avec edges")

NameError: name 'pcd_direct' is not defined

In [ ]:
color_linest = add_edges_between_layers(pts_top,pts)

NameError: name 'pts_top' is not defined

## Procédé 2 : Estimation de profondeur GLPN (heatmap) → nuage de points
Modèle GLPN pour estimer une carte de profondeur, puis création du nuage de points à partir de cette carte.

In [ ]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
import numpy as np
import open3d as o3d
import requests

Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']


In [ ]:
feature_extractor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")

Loading weights:   0%|          | 0/972 [00:00<?, ?it/s]

In [ ]:
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")

plt.imshow(image)
plt.show()

In [ ]:
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")
feature_extractor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")
inputs = feature_extractor(images=image, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)
    predicted_depth = outputs.predicted_depth

Loading weights:   0%|          | 0/972 [00:00<?, ?it/s]

In [ ]:
# Carte de profondeur GLPN (image conservee pour Procede 1)

pad = 16
depth_array = predicted_depth.squeeze().cpu().numpy() * 1000.0
depth_array = depth_array[pad:-pad, pad:-pad]
image_cropped = image.crop((pad, pad, image.width - pad, image.height - pad))

In [ ]:
def revert_depth_image(depth_image):
    """
    Inverse la profondeur de l'image : proche <-> lointain.
    depth_image : array numpy 2D (H, W), valeurs de profondeur.
    Retourne une copie avec depth_inv = depth_max - depth + depth_min (range préservé, ordre inversé).
    """
    d = np.asarray(depth_image, dtype=np.float64)
    d_min, d_max = d.min(), d.max()
    return (d_max - d + d_min).astype(depth_image.dtype if hasattr(depth_image, 'dtype') else np.float32)

In [ ]:
depth_array = revert_depth_image(depth_array)

In [ ]:
# Apercu Procede 2 : image rognee et carte de profondeur GLPN
fig, ax = plt.subplots(1, 2)
ax[0].imshow(image_cropped)
ax[0].set_title("Image rognee")
ax[1].imshow(depth_array, cmap='plasma')
ax[1].set_title("Profondeur GLPN")
ax[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.tight_layout()
plt.show()

In [ ]:
# Intrinsèques Procédé 2 : dimensions et caméra dérivées de l'image importée (rognée)
h2, w2 = image_cropped.height, image_cropped.width
intrinsic_2 = o3d.camera.PinholeCameraIntrinsic(w2, h2, 1000.0, 1000.0, w2 / 2, h2 / 2)

In [ ]:
def create_point_cloud_from_depth_image(depth_image, intrinsic, scale=1.0, color_image=None):
    """
    Nuage de points à partir d'une carte de profondeur (numpy 2D).
    Si color_image est fourni (PIL ou numpy H,W,3), les points gardent ces couleurs.
    """
    height, width = depth_image.shape
    depth_o3d = o3d.geometry.Image((depth_image / scale).astype(np.float32))
    if color_image is not None:
        color_np = np.asarray(color_image, dtype=np.uint8)
        if color_np.ndim == 2:
            color_np = np.stack([color_np] * 3, axis=-1)
        if color_np.shape[0] != height or color_np.shape[1] != width:
            pil_img = Image.fromarray(color_np).resize((width, height), Image.Resampling.LANCZOS)
            color_np = np.asarray(pil_img, dtype=np.uint8)
        color_o3d = o3d.geometry.Image(np.ascontiguousarray(color_np))
    else:
        color_o3d = o3d.geometry.Image(np.zeros((height, width, 3), dtype=np.uint8))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=1000.0, depth_trunc=1000.0, convert_rgb_to_intensity=False
    )
    return o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)

In [ ]:
def point_cloud_to_closed_mesh_poisson(
    pcd,
    depth=9,
    density_quantile=0.05,
    normal_radius=0.05,
    normal_max_nn=30,
):
    """
    Convertit un nuage de points (Open3D) en maillage fermé par reconstruction de Poisson.
    Estime les normales si absentes, puis filtre les sommets de faible densité.
    """
    if not pcd.has_normals():
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(
                radius=normal_radius, max_nn=normal_max_nn
            )
        )
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=depth
    )
    densities = np.asarray(densities)
    vertices_to_keep = densities > np.quantile(densities, density_quantile)
    mesh = mesh.select_by_index(np.where(vertices_to_keep)[0])
    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    return mesh

In [ ]:
def point_cloud_to_closed_mesh_alpha_shape(pcd, alpha, clean=True):
    """
    Convertit un nuage de points (Open3D) en maillage fermé par alpha shape (alpha wrapping).
    alpha : plus petit = surface plus détaillée, plus grand = surface plus lisse.
    clean : si True, supprime triangles dégénérés/dupliqués et recalcule les normales.
    """
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    if clean:
        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    return mesh

In [ ]:
# Nuage de points Procédé 2 (carte de profondeur GLPN + couleurs de l'image)
pcd_glpn = create_point_cloud_from_depth_image(depth_array, intrinsic_2, scale=1.0, color_image=image_cropped)
o3d.visualization.draw_geometries([pcd_glpn], window_name="Procede 2 - GLPN")

In [ ]:
# Nuage de points Procédé 2 → géométrie fermée (Poisson)
mesh_glpn = point_cloud_to_closed_mesh_poisson(pcd_glpn)
o3d.visualization.draw_geometries([mesh_glpn], window_name="Maillage fermé (Poisson) - pcd_glpn")

In [ ]:
# Même nuage → géométrie fermée par alpha wrapping (ajuster alpha au besoin)
mesh_glpn_alpha = point_cloud_to_closed_mesh_alpha_shape(pcd_glpn, alpha=0.03)
o3d.visualization.draw_geometries([mesh_glpn_alpha], window_name="Maillage fermé (Alpha shape) - pcd_glpn")

[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh
[Open3D WARNING] [CreateFromPointCloudAlphaShape] invalid tetra in TetraMesh

### Checkpoint Depth Pro — en cas d'erreur "failed finding central directory"

L'erreur **PytorchStreamReader failed reading zip archive: failed finding central directory** signifie que le fichier `depth_pro.pt` est **tronqué ou corrompu** (téléchargement interrompu ou fichier endommagé). Il faut le re-télécharger.

**Dans PowerShell (une seule fois) :**
```powershell
Invoke-WebRequest -Uri "https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt" `
  -OutFile "C:\Users\mvm\open3d_vision\ml-depth-pro\checkpoints\depth_pro_alt.pt" -UseBasicParsing
```
Puis utiliser ci-dessous `depth_pro_alt.pt` comme chemin de checkpoint.

In [7]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
import numpy as np
import open3d as o3d
import requests

Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']


In [8]:
import dataclasses
from pathlib import Path
import depth_pro
from depth_pro.depth_pro import create_model_and_transforms, DEFAULT_MONODEPTH_CONFIG_DICT

# Utiliser depth_pro_alt.pt après re-téléchargement (voir cellule markdown au-dessus si erreur "central directory")
CHECKPOINT = Path(r"C:\Users\mvm\open3d_vision\ml-depth-pro\checkpoints\depth_pro_alt.pt")
config = dataclasses.replace(DEFAULT_MONODEPTH_CONFIG_DICT, checkpoint_uri=str(CHECKPOINT))
model, transform = create_model_and_transforms(config=config)
model.eval()

image, _, f_px = depth_pro.load_rgb(r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg")
image = transform(image)
prediction = model.infer(image, f_px=f_px)
depth = prediction["depth"].squeeze().cpu().numpy()  # en mètres, numpy pour revert_depth_image()

In [9]:
def revert_depth_image(depth_image):
    """
    Inverse la profondeur de l'image : proche <-> lointain.
    depth_image : array numpy 2D (H, W), valeurs de profondeur.
    Retourne une copie avec depth_inv = depth_max - depth + depth_min (range préservé, ordre inversé).
    """
    d = np.asarray(depth_image, dtype=np.float64)
    d_min, d_max = d.min(), d.max()
    return (d_max - d + d_min).astype(depth_image.dtype if hasattr(depth_image, 'dtype') else np.float32)

In [10]:
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")
feature_extractor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")
inputs = feature_extractor(images=image, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)
    predicted_depth = outputs.predicted_depth

Loading weights:   0%|          | 0/972 [00:00<?, ?it/s]

In [11]:
pad = 16
depth_array = predicted_depth.squeeze().cpu().numpy() * 1000.0
depth_array = depth_array[pad:-pad, pad:-pad]
image_cropped = image.crop((pad, pad, image.width - pad, image.height - pad))

In [12]:
# Apercu Procede 2 : image rognee et carte de profondeur GLPN
depth = revert_depth_image(depth)
fig, ax = plt.subplots(1, 2)
ax[0].imshow(image_cropped)
ax[0].set_title("Image rognee")
ax[1].imshow(depth, cmap='plasma')
ax[1].set_title("Profondeur GLPN")
ax[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.tight_layout()
plt.show()

In [13]:
image = np.asarray(image)
h2, w2 = len(image[0]), len(image[0][0])
intrinsic_2 = o3d.camera.PinholeCameraIntrinsic(w2, h2, 1000.0, 1000.0, w2 / 2, h2 / 2)

In [14]:
def create_point_cloud_from_depth_image(depth_image, intrinsic, scale=1.0, color_image=None):
    """
    Nuage de points à partir d'une carte de profondeur (numpy 2D).
    Si color_image est fourni (PIL ou numpy H,W,3), les points gardent ces couleurs.
    """
    height, width = depth_image.shape
    depth_o3d = o3d.geometry.Image((depth_image / scale).astype(np.float32))
    if color_image is not None:
        color_np = np.asarray(color_image, dtype=np.uint8)
        if color_np.ndim == 2:
            color_np = np.stack([color_np] * 3, axis=-1)
        if color_np.shape[0] != height or color_np.shape[1] != width:
            pil_img = Image.fromarray(color_np).resize((width, height), Image.Resampling.LANCZOS)
            color_np = np.asarray(pil_img, dtype=np.uint8)
        color_o3d = o3d.geometry.Image(np.ascontiguousarray(color_np))
    else:
        color_o3d = o3d.geometry.Image(np.zeros((height, width, 3), dtype=np.uint8))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=1000.0, depth_trunc=1000.0, convert_rgb_to_intensity=False
    )
    return o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)

In [15]:
# Nuage de points Procédé 2 (carte de profondeur GLPN + couleurs de l'image)
# S'assurer que l'image a le bon format pour color_image (HxWx3, dtype=uint8)
if isinstance(image, torch.Tensor):
    image_np = image.squeeze().permute(1, 2, 0).cpu().numpy()
else:
    image_np = np.array(image)

# Normaliser l'image en uint8 et 3 canaux
if image_np.dtype != np.uint8:
    image_np = (image_np * 255).clip(0, 255).astype(np.uint8)
if image_np.ndim == 2:
    # Image N&B, on la convertit en RGB
    image_np = np.stack([image_np]*3, axis=-1)
if image_np.shape[2] == 1:
    image_np = np.repeat(image_np, 3, axis=2)
elif image_np.shape[2] > 3:
    image_np = image_np[..., :3]


In [16]:
def create_point_cloud_from_image(color_image, depth_image, intrinsic, depth_scale=1000.0, remove_white=True):
    """
    Nuage de points à partir d'une image couleur et d'une carte de profondeur (numpy).
    Utilise Open3D RGBDImage.create_from_color_and_depth + PointCloud.create_from_rgbd_image.
    Si remove_white=True, enlève les points complètement blancs (RGB ≈ 1,1,1).
    """
    h, w = depth_image.shape
    color_o3d = o3d.geometry.Image(np.asarray(color_image, dtype=np.uint8))
    depth_o3d = o3d.geometry.Image(depth_image.astype(np.float32))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=depth_scale, depth_trunc=1000.0, convert_rgb_to_intensity=False
    )
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    if remove_white and pcd.has_colors():
        colors = np.asarray(pcd.colors)
        not_white = np.any(colors < 0.65, axis=1)
        indices = np.where(not_white)[0]
        pcd = pcd.select_by_index(indices)
    return pcd

In [17]:
def create_normal_lines(pcd, normal_length=0.0001):
    """
    Crée un LineSet pour visualiser les normales d'un nuage de points.
    Chaque normale est représentée par une ligne partant du point.
    
    Parameters:
    - pcd: PointCloud Open3D avec des normales estimées
    - normal_length: Longueur des lignes de normales à afficher
    
    Returns:
    - LineSet représentant les normales
    """
    if not pcd.has_normals():
        return None
    
    pts = np.asarray(pcd.points)
    normals = np.asarray(pcd.normals)
    n = len(pts)
    
    # Créer les points de départ et d'arrivée pour chaque normale
    pts_start = pts
    pts_end = pts + normals * normal_length
    pts_all = np.vstack([pts_start, pts_end])
    
    # Créer les lignes : chaque ligne relie le point i au point i+n
    lines = np.array([[i, i + n] for i in range(n)], dtype=np.int32)
    
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(pts_all)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    # Couleur bleue pour les normales
    line_set.colors = o3d.utility.Vector3dVector(np.tile([[0, 0, 1]], (n, 1)))
    
    return line_set
points = create_point_cloud_from_depth_image(depth_array, intrinsic_2, scale=1.0, color_image=image_cropped)




In [18]:

# Estimation des normales
points.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.05, max_nn=30
    )
)

# Création des lignes de normales
normal_lines = create_normal_lines(points, normal_length=0.000005)

# Affichage avec les normales
geoms_to_show = [points]
if normal_lines is not None:
    geoms_to_show.append(normal_lines)

    
o3d.visualization.draw_geometries([points])

In [19]:

pcd_glpn = create_point_cloud_from_depth_image(depth, intrinsic_2, scale=1.0, color_image=image_np)
pcd_glpn.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.05, max_nn=30
    )
)

# Création des lignes de normales
normal_lines = create_normal_lines(pcd_glpn, normal_length=0.000005)
o3d.visualization.draw_geometries([pcd_glpn], window_name="Procede 2 - GLPN")

In [20]:
def point_cloud_to_closed_mesh_alpha_shape(pcd, alpha, clean=True):
    """
    Convertit un nuage de points (Open3D) en maillage fermé par alpha shape (alpha wrapping).
    alpha : plus petit = surface plus détaillée, plus grand = surface plus lisse.
    clean : si True, supprime triangles dégénérés/dupliqués et recalcule les normales.
    """
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    if clean:
        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    return mesh

Géométrie alpha shape

In [21]:
# Normales du nuage : estimation puis orientation cohérente (vers caméra au-dessus pour une pile vue du dessus)
if not pcd_glpn.has_normals():
    pcd_glpn.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
    )
center = np.asarray(pcd_glpn.points).mean(axis=0)
camera_above = center + np.array([0, 0, 1.0])  # au-dessus du nuage pour orienter les normales vers l'extérieur
pcd_glpn.orient_normals_towards_camera_location(camera_above)

tetra_mesh, pt_map = o3d.geometry.TetraMesh.create_from_point_cloud(pcd_glpn)
alpha = 0.000045
print(f"alpha={alpha:.6f}")
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
        pcd_glpn, alpha, tetra_mesh, pt_map)

# Ajuster les normales du mesh : orientation cohérente puis vers l'extérieur
mesh.compute_vertex_normals()
# La méthode 'orient_triangles_consistent()' n'existe pas dans Open3D.
# À la place, réordonner les triangles pour normaliser leur orientation (déjà fait ci-dessous)
# (aucune action ici, le code suivant gère manuellement l'orientation)
mesh_center = mesh.get_center()
verts = np.asarray(mesh.vertices)
tris = np.asarray(mesh.triangles)
for i in range(len(tris)):
    a, b, c = verts[tris[i]]
    n = np.cross(b - a, c - a)
    n /= (np.linalg.norm(n) + 1e-10)
    centroid = (a + b + c) / 3
    if np.dot(n, centroid - mesh_center) < 0:
        tris[i] = tris[i][[0, 2, 1]]
mesh.triangles = o3d.utility.Vector3iVector(tris)
mesh.compute_vertex_normals()


alpha=0.000045


TriangleMesh with 894062 points and 2103556 triangles.

In [22]:
voxel = mesh.voxelized(pitch=0.01)
mesh_closed = voxel.marching_cubes


AttributeError: 'open3d.cpu.pybind.geometry.TriangleMesh' object has no attribute 'voxelized'

In [26]:
o3d.visualization.draw_geometries([mesh])

In [24]:
o3d.io.write_point_cloud("dirt_pile_points.pcd", pcd_glpn)

True

In [25]:
o3d.io.write_triangle_mesh("dirt_pile.ply", mesh)

True